In [0]:
from pyspark.sql.functions import *

stores_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/stores/"

orders_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/orders/"

df_stores = spark.read.format("delta").load(stores_silver_path)

df_orders = spark.read.format("delta").load(orders_silver_path)

#Calculate store-level sales

In [0]:
store_sales = (
    df_orders
    .groupBy("store_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("total_amount").alias("total_sales"),
        avg("total_amount").alias("average_order_value")
    )
)

# Add store information

In [0]:
df_store_sales = (
    store_sales
    .join(
        df_stores.select(
            "store_id",
            "store_name",
            "city",
            "state"
        ),
        on="store_id",
        how="left"
    )
)

In [0]:
df_store_sales = df_store_sales.select(
    "store_id",
    "store_name",
    "city",
    "state",
    "total_orders",
    "total_quantity",
    "total_sales",
    "average_order_value"
)
display(df_store_sales)

# Write to Gold

In [0]:
store_sales_gold_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/gold/store_sales/"
df_store_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(store_sales_gold_path)

# Verify

In [0]:
df_store_sales_gold = (
    spark.read
    .format("delta")
    .load(store_sales_gold_path)
)

display(df_store_sales_gold)

print("Store Sales records:", df_store_sales_gold.count())